# Agentic Multi-Hop RAG — Colab Experiment
**Before running:** `Runtime → Change runtime type → A100 GPU`

In [22]:
# Cell 1: Mount Drive and set project path
from google.colab import drive
drive.mount('/content/drive')

import os

# Adjust if your Drive path is different
PROJECT_PATH = '/content/drive/MyDrive/CS572/Final'

os.chdir(PROJECT_PATH)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/CS572/Final
Files: ['IR Final Project.pdf', '.gitignore', 'README.md', 'requirements.txt', '.env', '.env.example', 'colab_experiment.ipynb', '.claude', 'tests', '.pytest_cache', '.git', 'src', 'docs', 'results']


In [23]:
# Cell 2: Install dependencies
!pip install -q -U transformers bitsandbytes accelerate
!pip install -q rank-bm25 datasets sentence-transformers python-dotenv
print('Done.')

Done.


In [24]:
# Cell 3: HuggingFace login
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(os.path.join(PROJECT_PATH, '.env'))
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)
print('Logged in to HuggingFace.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace.


In [25]:
# Cell 4: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


In [26]:
# Cell 5: Add project root to Python path
import sys, shutil

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

# Clear stale __pycache__ and cached modules
for root, dirs, _ in os.walk(PROJECT_PATH):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

print('Path set and cache cleared.')

Path set and cache cleared.


In [27]:
# Cell 6: Patch AgenticRAG and run_experiment to enforce min 2 hops (bypasses Drive sync issues)
import src.run_experiment as _run_exp
from src.agent import AgenticRAG, AGENT_PROMPT
import json, re

# --- Patch 1: AgenticRAG.answer with min_hops=2 and dedup-based stopping ---
def _new_answer(self, question):
    accumulated_context = []
    all_retrieved_docs = []
    per_hop_docs = []
    sub_queries = [question]
    current_query = question

    for hop in range(1, self.max_hops + 1):
        raw_docs = self.retriever.retrieve(current_query, top_k=self.top_k * 2)
        reranked_docs = self.reranker.rerank(current_query, raw_docs, top_k=self.top_k)
        all_retrieved_docs.extend(reranked_docs)
        per_hop_docs.append({'query': current_query, 'docs': reranked_docs})

        new_context = '\n'.join(d['text'] for d in reranked_docs)
        accumulated_context.append(f'[Hop {hop} — Query: {current_query}]\n{new_context}')

        prompt = AGENT_PROMPT.format(
            context='\n\n'.join(accumulated_context),
            question=question,
            hop=hop,
            max_hops=self.max_hops,
        )
        raw_response = self._call_llm(prompt)

        try:
            text = re.sub(r'```(?:json)?\s*', '', raw_response).strip().rstrip('`').strip()
            match = re.search(r'\{.*\}', text, re.DOTALL)
            decision = json.loads(match.group() if match else text)
        except (json.JSONDecodeError, ValueError, AttributeError):
            if hop >= self.min_hops:
                decision = {'sufficient': True, 'final_answer': ''}
            else:
                decision = {'sufficient': False, 'sub_query': question}

        next_query = decision.get('sub_query') or question
        # Also stop if LLM is proposing the same query again (no new info to retrieve)
        sufficient = decision.get('sufficient', False) or (
            hop >= self.min_hops and next_query.strip() == current_query.strip()
        )

        if sufficient and hop >= self.min_hops:
            final_answer = decision.get('final_answer', '')
            if not final_answer:
                context_str = '\n\n'.join(accumulated_context)
                final_answer = self._call_llm(
                    f'Answer concisely based on the context below.\n\n'
                    f'Context:\n{context_str}\n\nQuestion: {question}\n\nAnswer:'
                )
            return {
                'answer': final_answer,
                'num_hops': hop,
                'retrieved_docs': all_retrieved_docs,
                'per_hop_docs': per_hop_docs,
                'sub_queries': sub_queries,
            }

        sub_queries.append(next_query)
        current_query = next_query

    context_str = '\n\n'.join(accumulated_context)
    final_answer = self._call_llm(
        f'Answer concisely based on the context below.\n\n'
        f'Context:\n{context_str}\n\nQuestion: {question}\n\nAnswer:'
    )
    return {
        'answer': final_answer,
        'num_hops': self.max_hops,
        'retrieved_docs': all_retrieved_docs,
        'per_hop_docs': per_hop_docs,
        'sub_queries': sub_queries,
    }

AgenticRAG.answer = _new_answer
print('Patch 1: AgenticRAG.answer patched (min_hops=2, dedup stopping)')

# --- Patch 2: run_experiment._agent_answer_with_fallback ---
def _new_agent_answer_with_fallback(agent, question):
    try:
        return agent.answer(question)
    except Exception:
        reranked_docs = _run_exp._fallback_docs(agent, question)
        return {
            'answer': reranked_docs[0]['text'] if reranked_docs else '',
            'num_hops': 1,
            'retrieved_docs': reranked_docs,
            'per_hop_docs': [{'query': question, 'docs': reranked_docs}],
            'sub_queries': [question],
        }

_run_exp._agent_answer_with_fallback = _new_agent_answer_with_fallback
print('Patch 2: run_experiment._agent_answer_with_fallback patched (always calls agent.answer)')

Patch 1: AgenticRAG.answer patched (min_hops=2, dedup stopping)
Patch 2: run_experiment._agent_answer_with_fallback patched (always calls agent.answer)


In [28]:
# Cell 7: Pilot run — 5 samples
from src.run_experiment import run

pilot_results = run(num_samples=5)
print('Pilot results:', pilot_results)

Running baseline RAG on 5 samples...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

Running Agentic RAG on 5 samples...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok


=== RESULTS ===
Metric            Baseline    Agentic
-------------------------------------
EM                  0.0000     0.0000
F1                  0.0578     0.0419
MRR                 0.2667     0.2667
NDCG@10             0.3985     0.4012
avg_hops            1.0000     3.6000
Pilot results: {'baseline': {'EM': 0.0, 'F1': 0.057821908049756156, 'MRR': 0.26666666666666666, 'NDCG@10': 0.3984565739436487, 'avg_hops': 1.0}, 'agentic': {'EM': 0.0, 'F1': 0.04190476190476191, 'MRR': 0.26666666666666666, 'NDCG@10': 0.40123349334654373, 'avg_hops': 3.6, 'per_hop_MRR': 0.3158333333333333, 'per_hop_NDCG@10': 0.41882860041786774}}


In [29]:
# Cell 8: Full experiment — 50 samples (~60-90 min)
# Only run after pilot succeeds and avg_hops > 1.0 for agentic
from src.run_experiment import run

results = run(num_samples=50)
print('Final results:', results)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running baseline RAG on 50 samples...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok

Running Agentic RAG on 50 samples...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_tok


=== RESULTS ===
Metric            Baseline    Agentic
-------------------------------------
EM                  0.0000     0.1200
F1                  0.0748     0.2518
MRR                 0.3137     0.3388
NDCG@10             0.3880     0.4352
avg_hops            1.0000     3.7200
Final results: {'baseline': {'EM': 0.0, 'F1': 0.07482593136896401, 'MRR': 0.31366666666666665, 'NDCG@10': 0.38799738340253226, 'avg_hops': 1.0}, 'agentic': {'EM': 0.12, 'F1': 0.2518380303432613, 'MRR': 0.3388102453102453, 'NDCG@10': 0.43522341036478496, 'avg_hops': 3.72, 'per_hop_MRR': 0.3393611111111111, 'per_hop_NDCG@10': 0.42010689725588224}}


In [30]:
# Cell 9: Confirm results saved to Drive
import json

results_path = os.path.join(PROJECT_PATH, 'results', 'experiment_results.json')
with open(results_path) as f:
    saved = json.load(f)

print('Baseline:', saved['baseline'])
print('Agentic: ', saved['agentic'])
print()
print('Saved to Drive at:', results_path)

Baseline: {'EM': 0.0, 'F1': 0.07482593136896401, 'MRR': 0.31366666666666665, 'NDCG@10': 0.38799738340253226, 'avg_hops': 1.0}
Agentic:  {'EM': 0.12, 'F1': 0.2518380303432613, 'MRR': 0.3388102453102453, 'NDCG@10': 0.43522341036478496, 'avg_hops': 3.72, 'per_hop_MRR': 0.3393611111111111, 'per_hop_NDCG@10': 0.42010689725588224}

Saved to Drive at: /content/drive/MyDrive/CS572/Final/results/experiment_results.json
